In [ ]:
import json
from openai import OpenAI
from dotenv import load_dotenv
import os
from huggingface_hub import login

In [ ]:
# data = []

In [ ]:
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)

from datasets import load_dataset

ds = load_dataset("ayanmukherjee/repliqa-4-company-policies-answerable")

In [ ]:
for split in ds:
    print(split)

In [ ]:
# with open("question_answer.jsonl", "r") as f:
#     for i in range(200):
#         line = f.readline().strip()
#         line_json = json.loads(line)
#         question = line_json.get("question")
#         long_answer = line_json.get("long_answer")
#         data.append({"question": question, "answer": long_answer})

In [ ]:
# fine_tune_train = data[:100]
# fine_tune_val = data[101:151]
# fine_tune_test = data[151:]

In [ ]:
fine_tune_train = []
fine_tune_val = []
fine_tune_test = []

In [ ]:
for item in ds["train"]:
    fine_tune_train.append(item)

In [ ]:
for item in ds["test"]:
    fine_tune_test.append(item)


for item in ds["validation"]:
    fine_tune_val.append(item)

In [ ]:
fine_tune_val[0]

In [ ]:
def construct_msg(item):
    return [
        {"role": "user", "content": item["question"]},
        {"role": "assistant", "content": item["long_answer"]}
    ]

In [ ]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = construct_msg(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [ ]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [ ]:
write_jsonl(fine_tune_val, "jsonl/fine_tune_validation.jsonl")

In [ ]:
openai = OpenAI()

In [ ]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="company-policy-chatbot"
)

In [ ]:
OPENAI_FINE_TUNED_MODEL = "ft:gpt-4.1-nano-2025-04-14:personal:company-policy-chatbot:DY8296m5"

In [ ]:
def chat(question, history=[]):
    messages = [{"role": "user", "content": question}]
    response = openai.chat.completions.create(model= OPENAI_FINE_TUNED_MODEL, messages = messages)
    return response.choices[0].message.content

In [ ]:
print(chat(fine_tune_test[0]["question"]))

In [ ]:
fine_tune_test[0]["answer"]